# 05 — Customer Engagement Analysis

Builds the RFM foundation, then a transparent, documented engagement score, customer value segments, and lifecycle stages.

In [ ]:
import pandas as pd, numpy as np
tx = pd.read_csv("../outputs/cleaned/olist_customer_transactions.csv", parse_dates=["order_purchase_timestamp"])
REFERENCE_DATE = tx["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
print("Reference date:", REFERENCE_DATE.date())

Reference date: 2018-10-18

Reference date fixed at max(purchase date)+1 day, used consistently for every recency calculation to avoid data leakage (no future information beyond the dataset's own snapshot).

In [ ]:
order_level = tx.groupby(["customer_unique_id","order_id"], as_index=False).agg(
    order_purchase_timestamp=("order_purchase_timestamp","min"), order_value=("payment_value","first"))
rfm = order_level.groupby("customer_unique_id").agg(
    first_purchase_date=("order_purchase_timestamp","min"),
    last_purchase_date=("order_purchase_timestamp","max"),
    total_orders=("order_id","nunique"), total_spend=("order_value","sum")).reset_index()
rfm["recency_days"] = (REFERENCE_DATE - rfm["last_purchase_date"]).dt.days
rfm["average_order_value"] = rfm["total_spend"]/rfm["total_orders"]
print(f"{len(rfm):,} customers")

## Engagement score methodology

```
engagement_score = 100 x mean(
    minmax(total_orders),
    minmax(-recency_days),
    minmax(unique_categories)
)
```

Each component min-max normalized to [0,1] before combining; equal 1/3 weights (transparent default — no business-supplied weighting scheme). **Review-based signals excluded** — the reviews file was unavailable this run.

In [ ]:
def minmax(s):
    return (s-s.min())/(s.max()-s.min()) if s.max()!=s.min() else pd.Series(0.5, index=s.index)

unique_cats = tx.groupby("customer_unique_id")["product_category"].nunique().rename("unique_categories")
rfm = rfm.merge(unique_cats, on="customer_unique_id", how="left")

freq_n = minmax(rfm["total_orders"]); rec_n = minmax(-rfm["recency_days"]); breadth_n = minmax(rfm["unique_categories"])
rfm["engagement_score"] = ((freq_n + rec_n + breadth_n)/3*100).round(2)
rfm["engagement_score"].describe()

count    93358.000000
mean        22.55
std          7.41
min          0.00
25%         17.34
50%         23.33
75%         28.24
max         91.29

## Tercile-based engagement categorization

In [ ]:
q_low, q_high = rfm["engagement_score"].quantile([0.33, 0.67])
rfm["engagement_category"] = pd.cut(rfm["engagement_score"], bins=[-1,q_low,q_high,101],
                                     labels=["Low Engagement","Moderately Engaged","Highly Engaged"])
rfm["engagement_category"].value_counts()